# SPEC11 - trainability and the evaluation point (pilot)

**Upload** `spec11_bundle.zip` into this session's files (left panel, Files, upload; it lands in `/content`).
Nothing is installed; the notebook uses only Python, torch and numpy. It also runs unchanged on a CPU-only machine
(device auto-detected; outside Colab the working folder is the current directory).

Cells:
1. this text;
2. environment print (python, torch, CUDA, GPU, free disk); unzips the bundle to `spec11/` if not already there and
   verifies every file against `MANIFEST.json` (md5);
3. QUICK smoke run (`SPEC11_QUICK=1`, a few minutes) into `spec11_quick_out/`, then the report on it;
4. the full run into `spec11_out/`, output streamed line by line; every 10 minutes `units/` (not `vecs/`) is zipped to
   `spec11_out_partial.zip`;
5. the report on the full run;
6. zips `spec11_out/units`, `spec11_report.txt`, `spec11_results.json` and the final Part B vectors into
   `spec11_results.zip` and prints its path and size (download it from the Files panel).

**The run is resumable.** Every finished unit is stored once, atomically, with a signature; re-running cell 4 skips
everything already stored. After a disconnect, re-run cells 2 and 4 if the files survived; if the runtime was reset
(files lost), download `spec11_out_partial.zip` regularly - note that it holds `units/` only, and later units need the
reference vectors in `vecs/`, so a lost runtime means re-running the references.


In [ ]:
import os, sys, json, hashlib, shutil, zipfile, platform, subprocess, time
import torch
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
BUNDLE_ZIP = os.path.join(BASE, "spec11_bundle.zip")
BUNDLE = os.path.join(BASE, "spec11")
print("python", platform.python_version(), "| torch", torch.__version__, "| cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("free disk: %.1f GB" % (shutil.disk_usage(BASE).free / 1e9))
if not os.path.exists(os.path.join(BUNDLE, "MANIFEST.json")):
    with zipfile.ZipFile(BUNDLE_ZIP) as z:
        z.extractall(BUNDLE)
man = json.load(open(os.path.join(BUNDLE, "MANIFEST.json")))
bad = []
for rel, d in man.items():
    h = hashlib.md5()
    with open(os.path.join(BUNDLE, rel), "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    if h.hexdigest() != d["md5"] or os.path.getsize(os.path.join(BUNDLE, rel)) != d["size"]:
        bad.append(rel)
print(f"MANIFEST: {len(man)} files, {len(bad)} mismatches {bad}")
assert not bad, "bundle files differ from MANIFEST.json"


def run_script(script, env_extra, partial_zip=None, out_dir=None, every_s=600):
    # runs python -I <bundle>/<script> with the SPEC11_* variables, streaming its output line by line
    env = dict(os.environ, SPEC11_BUNDLE=BUNDLE, **env_extra)
    p = subprocess.Popen([sys.executable, "-I", "-u", os.path.join(BUNDLE, script)], env=env, cwd=BASE,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    last = time.time()
    for line in p.stdout:
        print(line, end="", flush=True)
        if partial_zip and time.time() - last >= every_s:
            zip_units(out_dir, partial_zip)
            last = time.time()
    p.wait()
    if partial_zip:
        zip_units(out_dir, partial_zip)
    print(f"[{script} exited with code {p.returncode}]")
    return p.returncode


def zip_units(out_dir, path):
    tmp = path + ".tmp"
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
        ud = os.path.join(out_dir, "units")
        if os.path.isdir(ud):
            for fn in sorted(os.listdir(ud)):
                if fn.endswith(".json"):
                    z.write(os.path.join(ud, fn), os.path.join("units", fn))
    os.replace(tmp, path)


In [ ]:
QUICK_OUT = os.path.join(BASE, "spec11_quick_out")
run_script("spec11_run.py", dict(SPEC11_OUT=QUICK_OUT, SPEC11_QUICK="1"))
run_script("spec11_report.py", dict(SPEC11_OUT=QUICK_OUT, SPEC11_QUICK="1"))


In [ ]:
OUT = os.path.join(BASE, "spec11_out")
run_script("spec11_run.py", dict(SPEC11_OUT=OUT), partial_zip=os.path.join(BASE, "spec11_out_partial.zip"), out_dir=OUT)


In [ ]:
run_script("spec11_report.py", dict(SPEC11_OUT=OUT))


In [ ]:
RES = os.path.join(BASE, "spec11_results.zip")
with zipfile.ZipFile(RES + ".tmp", "w", zipfile.ZIP_DEFLATED) as z:
    for fn in sorted(os.listdir(os.path.join(OUT, "units"))):
        z.write(os.path.join(OUT, "units", fn), os.path.join("spec11_out", "units", fn))
    for fn in ("spec11_report.txt", "spec11_results.json"):
        if os.path.exists(os.path.join(OUT, fn)):
            z.write(os.path.join(OUT, fn), os.path.join("spec11_out", fn))
    vd = os.path.join(OUT, "vecs")
    for fn in sorted(os.listdir(vd)) if os.path.isdir(vd) else []:
        if fn.startswith("B__") and fn.endswith("__vc.pt"):          # final Part B vectors
            z.write(os.path.join(vd, fn), os.path.join("spec11_out", "vecs", fn))
os.replace(RES + ".tmp", RES)
print(RES, "%.1f MB" % (os.path.getsize(RES) / 1e6))
